# 214 — Ingest Layer 2 into Neo4j

Loads all merged CSVs from `data/layer_2/` into Neo4j AuraDB as Layer 2 (regulatory) nodes,
then creates the bridge relationship to the existing Layer 1 entity graph.

| Nodes created | Source CSV |
|---|---|
| `:Regulation` | `regulations.csv` |
| `:Section` | `sections.csv` |
| `:Requirement` | `requirements.csv` |
| `:Threshold` | `thresholds.csv` |
| `:Chunk` | `chunks.csv` |

Re-runnable: yes — clears existing Layer 2 nodes before loading.

In [1]:
import sys, os, json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.graph.connection import Neo4jConnection

LAYER2_DIR       = project_root / 'data' / 'layer_2'
INTERMEDIATE_DIR = LAYER2_DIR / 'intermediate'

# Merged outputs written by notebooks 211-213 to data/layer_2/
regulations_df  = pd.read_csv(LAYER2_DIR / 'regulations.csv')  # written by 211
sections_df     = pd.read_csv(LAYER2_DIR / 'sections.csv')            # written by 212
requirements_df = pd.read_csv(LAYER2_DIR / 'requirements.csv')        # written by 212
thresholds_df   = pd.read_csv(LAYER2_DIR / 'thresholds.csv')          # written by 212
chunks_df       = pd.read_csv(LAYER2_DIR / 'chunks.csv')              # written by 213
cross_refs_df   = pd.read_csv(LAYER2_DIR / 'cross_references.csv')    # written by 212

print(f'Regulations  : {len(regulations_df)}')
print(f'Sections     : {len(sections_df)}')
print(f'Requirements : {len(requirements_df)}')
print(f'Thresholds   : {len(thresholds_df)}')
print(f'Chunks       : {len(chunks_df)}')
print(f'Cross-refs   : {len(cross_refs_df)}')


Regulations  : 3
Sections     : 106
Requirements : 226
Thresholds   : 248
Chunks       : 138
Cross-refs   : 23


## Step 1 — Connect and create constraints

In [2]:
conn = Neo4jConnection()
conn.connect()

result = conn.run_query("RETURN 'Connected' AS status")
print(result[0]['status'])

Connected


In [3]:
CONSTRAINTS = [
    'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Regulation)  REQUIRE n.regulation_id  IS UNIQUE',
    'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Section)     REQUIRE n.section_id     IS UNIQUE',
    'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Requirement) REQUIRE n.requirement_id IS UNIQUE',
    'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Threshold)   REQUIRE n.threshold_id   IS UNIQUE',
    'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Chunk)       REQUIRE n.chunk_id       IS UNIQUE',
]
for stmt in CONSTRAINTS:
    conn.run_query(stmt)
print(f'Applied {len(CONSTRAINTS)} constraints.')

Applied 5 constraints.


## Step 2 — Clear existing Layer 2 nodes

In [4]:
conn.run_query(
    'MATCH (n) WHERE n:Regulation OR n:Section OR n:Requirement OR n:Threshold OR n:Chunk '
    'DETACH DELETE n'
)
print('Layer 2 nodes cleared.')

Layer 2 nodes cleared.


## Step 3 — Load nodes

In [5]:
def clean_rows(df: pd.DataFrame) -> list:
    '''Convert DataFrame to list of dicts, replacing NaN with None.'''
    return df.where(pd.notnull(df), None).to_dict('records')


def load_nodes(df: pd.DataFrame, label: str, id_field: str, batch_size: int = 500) -> int:
    '''MERGE nodes in batches. Returns total rows written.'''
    records = clean_rows(df)
    cypher  = (
        f'UNWIND $rows AS row '
        f'MERGE (n:{label} {{{id_field}: row.{id_field}}}) '
        f'SET n += row'
    )
    total = 0
    for i in range(0, len(records), batch_size):
        conn.run_query(cypher, {'rows': records[i:i + batch_size]})
        total += len(records[i:i + batch_size])
    return total


# :Regulation
n = load_nodes(regulations_df, 'Regulation', 'regulation_id')
print(f'Regulation   : {n} nodes')

# :Section — drop heavy text columns from node properties; keep summary only
sec_cols = [c for c in sections_df.columns if c != 'content_summary' or True]  # keep all
n = load_nodes(sections_df, 'Section', 'section_id')
print(f'Section      : {n} nodes')

# :Requirement
n = load_nodes(requirements_df, 'Requirement', 'requirement_id')
print(f'Requirement  : {n} nodes')

# :Threshold — condition_context is stored as a JSON string
n = load_nodes(thresholds_df, 'Threshold', 'threshold_id')
print(f'Threshold    : {n} nodes')

# :Chunk — store text property, exclude embedding (added in 215)
chunk_load_df = chunks_df.drop(columns=['text'], errors='ignore')  # text stored separately
n = load_nodes(chunks_df[['chunk_id', 'section_id', 'token_count', 'chunk_index', 'source_document']], 'Chunk', 'chunk_id')
print(f'Chunk        : {n} nodes')

# Set text as a separate property (avoids SET n += row including large text in every batch comparison)
text_records = chunks_df[['chunk_id', 'text']].where(pd.notnull(chunks_df[['chunk_id', 'text']]), None).to_dict('records')
for i in range(0, len(text_records), 500):
    conn.run_query(
        'UNWIND $rows AS row MATCH (c:Chunk {chunk_id: row.chunk_id}) SET c.text = row.text',
        {'rows': text_records[i:i + 500]},
    )
print(f'Chunk text   : set on {len(text_records)} nodes')

Regulation   : 3 nodes
Section      : 106 nodes
Requirement  : 226 nodes
Threshold    : 248 nodes
Chunk        : 138 nodes
Chunk text   : set on 138 nodes


## Step 4 — Create relationships

In [6]:
def load_rels(df: pd.DataFrame, cypher: str, batch_size: int = 500) -> int:
    records = clean_rows(df)
    total   = 0
    for i in range(0, len(records), batch_size):
        conn.run_query(cypher, {'rows': records[i:i + batch_size]})
        total += len(records[i:i + batch_size])
    return total


# (Regulation)-[:CONTAINS_SECTION]->(Section)
n = load_rels(
    sections_df[['regulation_id', 'section_id']],
    'UNWIND $rows AS row '
    'MATCH (r:Regulation {regulation_id: row.regulation_id}) '
    'MATCH (s:Section    {section_id:    row.section_id}) '
    'MERGE (r)-[:CONTAINS_SECTION]->(s)',
)
print(f'CONTAINS_SECTION     : {n} rels')

# (Section)-[:CONTAINS_REQUIREMENT]->(Requirement)
n = load_rels(
    requirements_df[['section_id', 'requirement_id']],
    'UNWIND $rows AS row '
    'MATCH (s:Section     {section_id:     row.section_id}) '
    'MATCH (q:Requirement {requirement_id: row.requirement_id}) '
    'MERGE (s)-[:CONTAINS_REQUIREMENT]->(q)',
)
print(f'CONTAINS_REQUIREMENT : {n} rels')

# (Requirement)-[:DEFINES_THRESHOLD]->(Threshold)
n = load_rels(
    thresholds_df[['requirement_id', 'threshold_id']],
    'UNWIND $rows AS row '
    'MATCH (q:Requirement {requirement_id: row.requirement_id}) '
    'MATCH (t:Threshold   {threshold_id:   row.threshold_id}) '
    'MERGE (q)-[:DEFINES_THRESHOLD]->(t)',
)
print(f'DEFINES_THRESHOLD    : {n} rels')

# (Section)-[:HAS_CHUNK]->(Chunk)
n = load_rels(
    chunks_df[['section_id', 'chunk_id']],
    'UNWIND $rows AS row '
    'MATCH (s:Section {section_id: row.section_id}) '
    'MATCH (c:Chunk   {chunk_id:   row.chunk_id}) '
    'MERGE (s)-[:HAS_CHUNK]->(c)',
)
print(f'HAS_CHUNK            : {n} rels')

# (Chunk)-[:NEXT_CHUNK]->(Chunk) — sequential within each section
for section_id, grp in chunks_df.groupby('section_id'):
    ordered = grp.sort_values('chunk_index')['chunk_id'].tolist()
    pairs = [{'a': ordered[i], 'b': ordered[i + 1]} for i in range(len(ordered) - 1)]
    if pairs:
        conn.run_query(
            'UNWIND $rows AS row '
            'MATCH (a:Chunk {chunk_id: row.a}) '
            'MATCH (b:Chunk {chunk_id: row.b}) '
            'MERGE (a)-[:NEXT_CHUNK]->(b)',
            {'rows': pairs},
        )
next_count = conn.run_query('MATCH ()-[r:NEXT_CHUNK]->() RETURN count(r) AS n')[0]['n']
print(f'NEXT_CHUNK           : {next_count} rels')

# (Section)-[:CROSS_REFERENCES {reference_type, description, confidence}]->(Section)
if not cross_refs_df.empty:
    n = load_rels(
        cross_refs_df,
        'UNWIND $rows AS row '
        'MATCH (a:Section {section_id: row.from_section_id}) '
        'MATCH (b:Section {section_id: row.to_section_id}) '
        'MERGE (a)-[r:CROSS_REFERENCES]->(b) '
        'SET r.reference_type = row.reference_type, '
        '    r.description    = row.description, '
        '    r.confidence     = row.confidence',
    )
    print(f'CROSS_REFERENCES     : {n} rels')

CONTAINS_SECTION     : 106 rels
CONTAINS_REQUIREMENT : 226 rels
DEFINES_THRESHOLD    : 248 rels
HAS_CHUNK            : 138 rels
NEXT_CHUNK           : 93 rels
CROSS_REFERENCES     : 23 rels


## Step 5 — Connect to Layer 1 entity graph

In [7]:
# Link every Regulation to the Federal jurisdiction node (bridge to Layer 1)
result = conn.run_query(
    'MATCH (r:Regulation) '
    'MATCH (j:Jurisdiction {jurisdiction_id: $jid}) '
    'MERGE (r)-[:APPLIES_TO_JURISDICTION {scope: $scope}]->(j) '
    'RETURN count(r) AS linked',
    {'jid': 'JUR-AU-FED', 'scope': 'all_adis'},
)
print(f'APPLIES_TO_JURISDICTION: {result[0]["linked"]} regulation(s) linked to JUR-AU-FED')

APPLIES_TO_JURISDICTION: 3 regulation(s) linked to JUR-AU-FED


## Step 6 — Node and relationship counts

In [8]:
node_labels = ['Regulation', 'Section', 'Requirement', 'Threshold', 'Chunk']
rel_types   = [
    'CONTAINS_SECTION', 'CONTAINS_REQUIREMENT', 'DEFINES_THRESHOLD',
    'HAS_CHUNK', 'NEXT_CHUNK', 'CROSS_REFERENCES', 'APPLIES_TO_JURISDICTION',
]

print('Node counts:')
for label in node_labels:
    n = conn.run_query(f'MATCH (n:{label}) RETURN count(n) AS cnt')[0]['cnt']
    print(f'  {label:<20} : {n}')

print('\nRelationship counts:')
for rel in rel_types:
    n = conn.run_query(f'MATCH ()-[r:{rel}]->() RETURN count(r) AS cnt')[0]['cnt']
    print(f'  {rel:<25} : {n}')

conn.close()
print('\nConnection closed.')

Node counts:
  Regulation           : 3
  Section              : 106
  Requirement          : 201
  Threshold            : 139
  Chunk                : 138

Relationship counts:
  CONTAINS_SECTION          : 106
  CONTAINS_REQUIREMENT      : 226
  DEFINES_THRESHOLD         : 248
  HAS_CHUNK                 : 138
  NEXT_CHUNK                : 93
  CROSS_REFERENCES          : 23
  APPLIES_TO_JURISDICTION   : 3

Connection closed.
